# GraphRAG-QPP: Benchmark Execution Notebook

This notebook allows you to run the complete GraphRAG-QPP benchmark pipeline directly on **Google Colab (L4 / A100 GPU)** or a local GPU workstation.

### Pipeline Overview
1. **Datasets**: LongMemEval-S (conversational memory), HotpotQA (multi-hop QA), MuSiQue (complex multi-hop QA).
2. **Production Pairing (ADR-0004)**: Qwen3-Embedding (dense retriever) + RedHatAI/Qwen3.5-9B-FP8-dynamic (generator) + google/gemma-4-12B-it-qat-w4a16-ct (judge).
3. **Topological Context Selectors (ADR-0003 & ADR-0006)**: 2-layer GATv2 GNN and Topological LightGBM under Conservative Graph Abstention (ADR-0005).
4. **Artifacts**: Cached Parquet results, PyTorch `.pt` candidate subgraphs, publication LaTeX tables, and diagnostic figures.

## 1. Hardware Verification

In [ ]:
!nvidia-smi

## 2. Environment Setup & Dependency Installation

In [ ]:
# If running on Colab, clone the repository (or extract uploaded files)
import os
if not os.path.exists('/content/graphrag-qpp'):
    !git clone https://github.com/AmirBnsL/graphrag-qpp.git /content/graphrag-qpp
%cd /content/graphrag-qpp

# Install dependencies in editable mode
!pip install -q -e . vllm torch-geometric sentence-transformers lightgbm datasets

## 3. Fast Validation Pilot (20 Queries)
Run this first to verify that all models, embeddings, and automated assertions pass before running the full 1,000 queries.

In [ ]:
!python scripts/run_benchmarks.py \
    --mode colab \
    --max-queries-lme 10 \
    --max-queries-multihop 10 \
    --output-dir artifacts/pilot_v2

## 4. Full Benchmark Execution (1,000 Queries)
Executes the full experimental workload: 500 LongMemEval-S + 250 HotpotQA + 250 MuSiQue (~60–75 minutes on L4 GPU).

In [ ]:
!python scripts/run_benchmarks.py \
    --mode colab \
    --max-queries-lme 500 \
    --max-queries-multihop 500 \
    --output-dir artifacts/full

## 5. View Generated Diagnostic Figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('artifacts/full/figures') if Path('artifacts/full/figures').exists() else Path('artifacts/pilot_v2/figures')
for p in sorted(fig_dir.glob('*.png')):
    print(f"\n=== {p.name} ===")
    display(Image(str(p)))